# Leitura dos Dados

In [3]:
import pandas as pd
arquivos = [
    'dim_clientes',  
    'dim_geografia',  
    'dim_produtos',
    'dim_tradução_produto',
    'dim_vendedores',  
    'fact_avaliações',
    'fact_itens',
    'fact_pagamentos',            
    'fact_pedidos',
]   
dado_cru = {}        
for nome in arquivos:
    dado_cru[nome] = pd.read_csv(rf"C:\Users\daniel-pc\Documents\Projetos\E-commerce\{nome}.csv")
# Atribuição de variáveis
clientes = dado_cru['dim_clientes']
geografia = dado_cru['dim_geografia']
produtos = dado_cru['dim_produtos']      
tradução_produtos = dado_cru['dim_tradução_produto']      
vendedores = dado_cru['dim_vendedores']      
pedidos = dado_cru['fact_avaliações']      
itens = dado_cru['fact_itens']      
pagamentos = dado_cru['fact_pagamentos']
pedidos = dado_cru['fact_pedidos']      



#### Investigação e limpeza da tabela clientes

In [4]:
print('Dados Crus:')
print(clientes.head(5))
# Mapeia 20 principais cidades do Brasil (20/80 [Principio de Pareto])
mapa_pareto_esmagado = {
    'saopaulo': 'São Paulo',
    'riodejaneiro': 'Rio de Janeiro',
    'belohorizonte': 'Belo Horizonte',
    'brasilia': 'Brasília',
    'curitiba': 'Curitiba',
    'campinas': 'Campinas',
    'portoalegre': 'Porto Alegre',
    'salvador': 'Salvador',
    'guarulhos': 'Guarulhos',
    'saobernardocampo': 'São Bernardo do Campo',
    'niteroi': 'Niterói',
    'santoandre': 'Santo André',
    'osasco': 'Osasco',
    'goiania': 'Goiânia',
    'saojosedoscampos': 'São José dos Campos',
    'recife': 'Recife',
    'fortaleza': 'Fortaleza',
    'sorocaba': 'Sorocaba',
    'ribeiraopreto': 'Ribeirão Preto',
    'florianopolis': 'Florianópolis'
}

def padronizar_cidade_br(nome_cru):
    texto_limpo = str(nome_cru).lower().strip()
    
    # Junta tudo sem espaço nenhum
    # Exemplo: " sao   paulo " -> split vira ['sao', 'paulo'] -> join vira "saopaulo"
    digital = ''.join(texto_limpo.split())
    
    # 1. Consulta pela digital no VIP
    if digital in mapa_pareto_esmagado:
        return mapa_pareto_esmagado[digital]
    
    # 2. Se não achou no VIP, aplica a regra gramatical normal usando o texto original
    preposicoes_br = {'de', 'da', 'do', 'das', 'dos', 'd', 'e', 'del'}
    palavras = texto_limpo.split()
    
    formatadas = []
    for i, palavra in enumerate(palavras):
        if i > 0 and palavra in preposicoes_br:
            formatadas.append(palavra)
        else:
            formatadas.append(palavra.capitalize())
            
    return ' '.join(formatadas)

clientes['customer_city'] = clientes['customer_city'].apply(padronizar_cidade_br)

print('Dados Transformados:')
print(clientes['customer_city'].value_counts().head(10))

# Salvamento do arquivo
caminho_clientes = rf"C:\Users\daniel-pc\Documents\Projetos\E-commerce\dados_limpos\dim_clientes_limpa.csv"
clientes.to_csv(caminho_clientes, index=False, encoding='utf-8')



Dados Crus:
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 Franca             SP  
1                      9790  Sao Bernardo do Campo             SP  
2                      1151              São Paulo             SP  
3                      8775        Mogi das Cruzes             SP  
4                     13056               Campinas             SP  
Dados Transformados:
customer_city
São Paulo                15540
Rio de Janeiro            6882
Belo Horizonte            2773
Brasília     

#### Investigação e limpeza da tabela geografia

In [5]:
print(f"Linhas originais brutas: {len(geografia)}")
geo_sem_clones = geografia.drop_duplicates()

# COMPRESSÃO (Pega coordenadas de um mesmo CEP e agrupa pela mediana de cordenadas)
dim_geo_compressa = geo_sem_clones.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean',
    'geolocation_city': 'first',
    'geolocation_state': 'first'
}).reset_index()

print(f"Linhas após compressão: {len(dim_geo_compressa)}")

# Aplica função na tabela COMPRIMIDA
dim_geo_compressa['geolocation_city'] = dim_geo_compressa['geolocation_city'].apply(padronizar_cidade_br)

caminho_salvar = rf"C:\Users\daniel-pc\Documents\Projetos\E-commerce\dados_limpos\dim_geografia_limpa.csv"

dim_geo_compressa.to_csv(caminho_salvar, index=False, encoding='utf-8') 



Linhas originais brutas: 1000163
Linhas após compressão: 19015


#### Investigação e limpeza da tabela produtos

In [6]:
print(produtos.info())
print(f"Duplicatas: {produtos.duplicated().sum()}")
produtos.head()

# CORREÇÃO GRAMATICAL DO INGLÊS (Consertando o "lenght")
produtos = produtos.rename(
    columns={
        "product_name_lenght": "product_name_length",
        "product_description_lenght": "product_description_length",
    }
)

# Tratamento de produtos com categoria vazia
produtos['product_category_name'] = produtos['product_category_name'].fillna("outros")

# Tratamento de fotos (Faltam 610 também. Quem não tem categoria, não tem foto)
# Preenche nulos com 0 e transforma de Float (2.0) para Inteiro (2)
produtos["product_photos_qty"] = (
    produtos["product_photos_qty"].fillna(0).astype(int)
)

# Cria dicionario para repetição futura
colunas_fisicas = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]

for col in colunas_fisicas:
    mediana_segura = produtos[col].median() # Pega mediana para todas as colunas fisicas 
    produtos[col] = produtos[col].fillna(mediana_segura) # Preenche nulos das colunas com mediana

caminho_salvar = rf"C:\Users\daniel-pc\Documents\Projetos\E-commerce\dados_limpos\dim_produtos_limpa.csv"
produtos.to_csv(caminho_salvar, index=False, encoding='utf-8')
produtos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB
None
Duplicatas: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                  

#### Investigação e limpeza da tabela vendedores

In [7]:
vendedores.head()
vendedores['seller_city'] = vendedores['seller_city'].apply(padronizar_cidade_br)
vendedores.head()

caminho_salvar = rf"C:\Users\daniel-pc\Documents\Projetos\E-commerce\dados_limpos\dim_vendedores_limpa.csv"
vendedores.to_csv(caminho_salvar, index=False, encoding='utf-8')

#### Investigação e limpeza da tabela avaliações


In [8]:
# Conversão de nulos dos comentários para "Nenhum"
pedidos['review_comment_title'] = pedidos['review_comment_title'].fillna('Nenhum')
pedidos['review_comment_message'] = pedidos['review_comment_message'].fillna('Nenhum')

# Conversão de tipo objeto para tipo data
pedidos["review_creation_date"] = pd.to_datetime(
    pedidos["review_creation_date"]
)
pedidos["review_answer_timestamp"] = pd.to_datetime(
    pedidos["review_answer_timestamp"]
)

pedidos.info()
pedidos.head()
caminho_salvar = rf"C:\Users\daniel-pc\Documents\Projetos\E-commerce\dados_limpos\fact_avaliações_limpa.csv"
pedidos.to_csv(caminho_salvar, index=False, encoding='utf-8')

KeyError: 'review_comment_title'

#### Investigação e limpeza da tabela itens

In [ ]:
itens["shipping_limit_date"] = pd.to_datetime(
    itens["shipping_limit_date"]
)
itens.info()
caminho_salvar = rf"C:\Users\daniel-pc\Documents\Projetos\E-commerce\dados_limpos\fact_itens_limpa.csv"
itens.to_csv(caminho_salvar, index=False, encoding='utf-8')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  object        
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  object        
 3   seller_id            112650 non-null  object        
 4   shipping_limit_date  112650 non-null  datetime64[ns]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(3)
memory usage: 6.0+ MB


#### Investigação e limpeza da tabela pagamentos


In [ ]:
# VALORES NULOS (Dinheiro sumido)
print("1. Valores nulos por coluna:")
print(pagamentos.isnull().sum())

# O que os clientes usaram para pagar?
print("\n2. Contagem por Método de Pagamento:")
print(pagamentos['payment_type'].value_counts())

# Alguém pagou em 0x ou em número negativo?)
parcelas_estranhas = pagamentos[pagamentos['payment_installments'] < 1]
print(f"\n3. Pagamentos com menos de 1 parcela: {len(parcelas_estranhas)} linhas")

# VALORES PAGOS ZERADOS OU NEGATIVOS
pagamento_gratis = pagamentos[pagamentos['payment_value'] <= 0]
print(f"4. Pagamentos com valor R$ 0,00 ou negativo: {len(pagamento_gratis)} linhas")


# Elimina as 3 transações fantasmas 'not_defined'
pagamentos_limpos = pagamentos[pagamentos['payment_type'] != 'not_defined'].copy()

# Corrige a aberração matemática de "0 parcelas" para à vista (1x)
pagamentos_limpos.loc[pagamentos_limpos['payment_installments'] == 0, 'payment_installments'] = 1

# Garante que as parcelas sejam inteiras (int)
pagamentos_limpos['payment_installments'] = pagamentos_limpos['payment_installments'].astype(int)

print(f"Linhas antes: {len(pagamentos)} | Linhas depois: {len(pagamentos_limpos)}")

caminho_salvar = rf"C:\Users\daniel-pc\Documents\Projetos\E-commerce\dados_limpos\fact_pagamentos_limpa.csv"
pagamentos.to_csv(caminho_salvar, index=False, encoding='utf-8')


1. Valores nulos por coluna:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

2. Contagem por Método de Pagamento:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

3. Pagamentos com menos de 1 parcela: 2 linhas
4. Pagamentos com valor R$ 0,00 ou negativo: 9 linhas
Linhas antes: 103886 | Linhas depois: 103883


#### Investigação e limpeza da tabela pedidos

In [16]:
print(pedidos.isnull().sum())
# Conversão de tipo objeto para tipo data
pedidos["order_delivered_carrier_date"] = pd.to_datetime(
    pedidos["order_delivered_carrier_date"]
)
pedidos["order_delivered_customer_date"] = pd.to_datetime(
    pedidos["order_delivered_customer_date"]
)
pedidos["order_estimated_delivery_date"] = pd.to_datetime(
    pedidos["order_estimated_delivery_date"]
)
pedidos["order_approved_at"] = pd.to_datetime(
    pedidos["order_approved_at"]
)
pedidos["order_purchase_timestamp"] = pd.to_datetime(
    pedidos["order_purchase_timestamp"]
)

pedidos_sem_data = pedidos[pedidos['order_delivered_customer_date'].isnull()]['order_status'].value_counts()

print("Quem são os 2.965 pedidos sem data de entrega?")
print(pedidos_sem_data)

mascarar_entregue_sem_aprovação = (pedidos['order_status'] == 'delivered') & (pedidos['order_approved_at'].isnull())
pedidos.loc[mascarar_entregue_sem_aprovação, 'order_approved_at'] = pedidos.loc[mascarar_entregue_sem_aprovação, 'order_purchase_timestamp']

mascarar_entregue_sem_data = (pedidos['order_status'] == 'delivered') & (pedidos['order_delivered_customer_date'].isnull())
pedidos.loc[mascarar_entregue_sem_data, 'order_delivered_customer_date'] = pedidos.loc[mascarar_entregue_sem_data, 'order_estimated_delivery_date']

print(pedidos.isnull().sum())
caminho_salvar = rf"C:\Users\daniel-pc\Documents\Projetos\E-commerce\dados_limpos\fact_pedidos_limpa.csv"
pedidos.to_csv(caminho_salvar, index=False, encoding='utf-8')

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 146
order_delivered_carrier_date     1783
order_delivered_customer_date    2957
order_estimated_delivery_date       0
dtype: int64
Quem são os 2.965 pedidos sem data de entrega?
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
created           5
approved          2
Name: count, dtype: int64
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 146
order_delivered_carrier_date     1783
order_delivered_customer_date    2957
order_estimated_delivery_date       0
dtype: int64
